# Семинар 4. Деревья решений

В этом семинаре мы разберем:
- Построение дерева решений вручную (энтропия, Gini, Information Gain)
- Деревья решений в sklearn: классификация, регрессия, decision boundary
- Сравнение критериев разбиения (Entropy vs Gini)
- Гиперпараметры, переобучение и pruning
- Нестабильность деревьев (мотивация для ансамблей)
- Isolation Forest (обнаружение аномалий)
- Инкрементальные деревья (Hoeffding Tree, concept drift)
- Применение: кредитный скоринг

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q scikit-learn river

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from matplotlib.colors import ListedColormap
from sklearn.datasets import load_iris, make_blobs, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, confusion_matrix

## 1. Деревья решений: теория

Ключевые алгоритмы:

- **ID3**: использует энтропию и прирост информации
- **C4.5**: улучшение ID3, может работать с непрерывными признаками
- **CART**: бинарное дерево, использует индекс Джини для классификации

Критерии разбиения:

- Энтропия: $H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$
- Индекс Джини: $G(S) = 1 - \sum_{i=1}^{c} p_i^2$

### 1.1 Построение дерева вручную

Классический пример: датасет "играть ли в теннис". Рассчитаем Information Gain для каждого признака и определим лучшее первое разбиение.

In [ ]:
data = {
    'Погода': ['Солнечно', 'Солнечно', 'Пасмурно', 'Дождь', 'Дождь', 'Дождь', 'Пасмурно', 'Солнечно', 'Солнечно', 'Дождь', 'Солнечно', 'Пасмурно', 'Пасмурно', 'Дождь'],
    'Температура': ['Жарко', 'Жарко', 'Жарко', 'Умеренно', 'Прохладно', 'Прохладно', 'Прохладно', 'Умеренно', 'Прохладно', 'Умеренно', 'Умеренно', 'Умеренно', 'Жарко', 'Умеренно'],
    'Влажность': ['Высокая', 'Высокая', 'Высокая', 'Высокая', 'Нормальная', 'Нормальная', 'Нормальная', 'Высокая', 'Нормальная', 'Нормальная', 'Нормальная', 'Высокая', 'Нормальная', 'Высокая'],
    'Ветер': ['Нет', 'Да', 'Нет', 'Нет', 'Нет', 'Да', 'Да', 'Нет', 'Нет', 'Нет', 'Да', 'Да', 'Нет', 'Да'],
    'Играть в теннис': ['Нет', 'Нет', 'Да', 'Да', 'Да', 'Нет', 'Да', 'Нет', 'Да', 'Да', 'Да', 'Да', 'Да', 'Нет'],
}

df = pd.DataFrame(data)
df

In [ ]:
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    ent = 0
    for i in range(len(elements)):
        prob = counts[i] / sum(counts)
        ent += -prob * np.log2(prob)
    return ent

def information_gain(data, split_attribute, target_attribute="Играть в теннис"):
    total_entropy = entropy(data[target_attribute])
    vals, counts = np.unique(data[split_attribute], return_counts=True)
    weighted_entropy = 0
    for i in range(len(vals)):
        subset_entropy = entropy(
            data.where(data[split_attribute] == vals[i]).dropna()[target_attribute]
        )
        weighted_entropy += (counts[i] / sum(counts)) * subset_entropy
    return total_entropy - weighted_entropy

for attribute in ['Погода', 'Температура', 'Влажность', 'Ветер']:
    print(f"Прирост информации для {attribute}: {information_gain(df, attribute):.4f}")

print("\nЛучший атрибут для первого разбиения - тот, для которого прирост информации максимален.")

## 2. Деревья решений в sklearn

### 2.1 Классификация

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

print(f"Точность на обучающей выборке: {clf.score(X_train, y_train):.4f}")
print(f"Точность на тестовой выборке: {clf.score(X_test, y_test):.4f}")

plt.figure(figsize=(20, 10))
plot_tree(clf, filled=True, feature_names=feature_names, class_names=class_names, rounded=True)
plt.title("Дерево решений для классификации ирисов")
plt.show()

importance = clf.feature_importances_
for i, feature in enumerate(feature_names):
    print(f"Важность признака '{feature}': {importance[i]:.4f}")

### 2.2 Decision boundary: прямоугольное разбиение пространства

Деревья решений разбивают пространство признаков на прямоугольные области (axis-aligned splits). С увеличением глубины дерева разбиение становится все более мелким.

In [ ]:
X_moon, y_moon = make_moons(n_samples=300, noise=0.25, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
cmap_bg = ListedColormap(['#FFAAAA', '#AAAAFF'])

for ax, depth in zip(axes, [1, 3, 10, None]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_moon, y_moon)
    eps = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X_moon[:, 0].min() - eps, X_moon[:, 0].max() + eps, 300),
        np.linspace(X_moon[:, 1].min() - eps, X_moon[:, 1].max() + eps, 300),
    )
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.pcolormesh(xx, yy, Z, cmap=cmap_bg, alpha=0.4)
    ax.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap='bwr', edgecolors='k', s=20)
    label = f'max_depth={depth}' if depth else 'max_depth=None'
    ax.set_title(f'{label}\nacc={tree.score(X_moon, y_moon):.2f}')
    ax.grid(alpha=0.2)

plt.suptitle('Decision boundary: деревья разбивают пространство на прямоугольники', fontsize=14)
plt.tight_layout()
plt.show()

### 2.3 Entropy vs Gini

Два основных критерия разбиения дают похожие, но не идентичные деревья. На практике разница обычно минимальна. Gini чуть быстрее (нет логарифма).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 8))

for ax, criterion in zip(axes, ['gini', 'entropy']):
    tree = DecisionTreeClassifier(max_depth=3, criterion=criterion, random_state=42)
    tree.fit(X_train, y_train)
    plot_tree(tree, filled=True, feature_names=feature_names,
              class_names=class_names, rounded=True, ax=ax)
    ax.set_title(f'criterion={criterion}, test acc={tree.score(X_test, y_test):.3f}')

plt.tight_layout()
plt.show()

### 2.4 Регрессия (DecisionTreeRegressor)

Деревья решений могут решать и задачу регрессии. Вместо класса в листе хранится среднее значение таргета. Критерий разбиения - минимизация MSE.

In [ ]:
# Зашумленная синусоида
np.random.seed(42)
X_reg = np.sort(np.random.uniform(0, 2 * np.pi, 200))[:, np.newaxis]
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.2, 200)

X_plot = np.linspace(0, 2 * np.pi, 500)[:, np.newaxis]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, depth in zip(axes, [2, 5, 20]):
    reg = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_reg, y_reg)
    y_pred = reg.predict(X_plot)
    ax.scatter(X_reg, y_reg, s=10, alpha=0.5, label='data')
    ax.plot(X_plot, np.sin(X_plot), 'g-', label='true', linewidth=2)
    ax.plot(X_plot, y_pred, 'r-', label='tree', linewidth=2)
    ax.set_title(f'max_depth={depth}, MSE={mean_squared_error(y_reg, reg.predict(X_reg)):.4f}')
    ax.legend()
    ax.grid(alpha=0.2)

plt.suptitle('Decision Tree Regression: ступенчатая аппроксимация', fontsize=14)
plt.tight_layout()
plt.show()

Дерево с depth=2 - underfitting (слишком грубая аппроксимация). Depth=20 - overfitting (повторяет шум). Оптимум где-то посередине.

## 3. Гиперпараметры и переобучение

### 3.1 Влияние глубины

In [ ]:
max_depths = np.arange(1, 20)
train_scores = []
test_scores = []

for depth in max_depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    test_scores.append(clf.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(max_depths, train_scores, 'o-', label='Обучающая выборка')
plt.plot(max_depths, test_scores, 'o-', label='Тестовая выборка')
plt.xlabel('Максимальная глубина дерева')
plt.ylabel('Точность')
plt.title('Влияние глубины дерева на точность')
plt.legend()
plt.grid(True)
plt.show()

### 3.2 GridSearchCV

In [ ]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7, 8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True,
)
grid_search.fit(X, y)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучшая точность: {grid_search.best_score_:.4f}")

results = grid_search.cv_results_
for depth in [3, 5, 7]:
    mask = results['param_max_depth'] == depth
    plt.figure(figsize=(10, 6))
    for split in [2, 5, 10]:
        split_mask = mask & (results['param_min_samples_split'] == split)
        plt.plot(
            results['param_min_samples_leaf'][split_mask],
            results['mean_test_score'][split_mask],
            'o-', label=f'min_samples_split={split}',
        )
    plt.xlabel('min_samples_leaf')
    plt.ylabel('Score')
    plt.title(f'Точность при max_depth={depth}')
    plt.legend()
    plt.grid(True)
    plt.show()

### 3.3 Pruning (cost-complexity pruning)

Вместо ручного подбора max_depth можно вырастить полное дерево и затем обрезать его (post-pruning). Параметр `ccp_alpha` контролирует силу обрезки: чем больше alpha, тем сильнее обрезка.

In [ ]:
# Строим путь обрезки
clf_full = DecisionTreeClassifier(random_state=42)
clf_full.fit(X_train, y_train)
path = clf_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

# Обучаем дерево для каждого alpha
train_scores_prune = []
test_scores_prune = []
n_leaves = []

for alpha in ccp_alphas:
    clf_pruned = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    clf_pruned.fit(X_train, y_train)
    train_scores_prune.append(clf_pruned.score(X_train, y_train))
    test_scores_prune.append(clf_pruned.score(X_test, y_test))
    n_leaves.append(clf_pruned.get_n_leaves())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(ccp_alphas, train_scores_prune, 'o-', label='Train', markersize=3)
axes[0].plot(ccp_alphas, test_scores_prune, 'o-', label='Test', markersize=3)
axes[0].set_xlabel('ccp_alpha')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs ccp_alpha')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(ccp_alphas, n_leaves, 'o-', markersize=3)
axes[1].set_xlabel('ccp_alpha')
axes[1].set_ylabel('Number of leaves')
axes[1].set_title('Tree complexity vs ccp_alpha')
axes[1].grid(True)

plt.tight_layout()
plt.show()

best_idx = np.argmax(test_scores_prune)
print(f"Лучший ccp_alpha: {ccp_alphas[best_idx]:.4f}")
print(f"Test accuracy: {test_scores_prune[best_idx]:.4f}")
print(f"Листьев: {n_leaves[best_idx]}")

## 4. Нестабильность деревьев

Деревья решений очень чувствительны к данным: удаление нескольких точек может полностью изменить структуру дерева. Это одна из главных причин, почему используют ансамбли (Random Forest, Gradient Boosting) - они усредняют множество нестабильных деревьев.

In [ ]:
# Обучаем дерево на полных данных и на данных с удаленными 3 точками
X_iris2 = iris.data[:, :2]  # берем 2 признака для визуализации
y_iris2 = iris.target

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
cmap_bg = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])

eps = 0.5
xx, yy = np.meshgrid(
    np.linspace(X_iris2[:, 0].min() - eps, X_iris2[:, 0].max() + eps, 300),
    np.linspace(X_iris2[:, 1].min() - eps, X_iris2[:, 1].max() + eps, 300),
)

for ax_idx, (title, remove_idx) in enumerate([
    ('Все данные (150 точек)', []),
    ('Удалены 3 точки из класса 1', [50, 55, 60]),
    ('Удалены 3 точки из класса 2', [100, 110, 120]),
]):
    mask = np.ones(len(X_iris2), dtype=bool)
    mask[remove_idx] = False
    X_sub, y_sub = X_iris2[mask], y_iris2[mask]
    
    tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_sub, y_sub)
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[ax_idx].pcolormesh(xx, yy, Z, cmap=cmap_bg, alpha=0.3)
    axes[ax_idx].scatter(X_sub[:, 0], X_sub[:, 1], c=y_sub, cmap='brg', edgecolors='k', s=30)
    if remove_idx:
        axes[ax_idx].scatter(X_iris2[remove_idx, 0], X_iris2[remove_idx, 1],
                            c='black', marker='x', s=100, linewidths=2, label='removed')
        axes[ax_idx].legend()
    axes[ax_idx].set_title(title)
    axes[ax_idx].set_xlabel(feature_names[0])
    axes[ax_idx].set_ylabel(feature_names[1])
    axes[ax_idx].grid(alpha=0.2)

plt.suptitle('Нестабильность деревьев: удаление 3 точек меняет границы', fontsize=14)
plt.tight_layout()
plt.show()

Видно, что удаление всего 3 из 150 точек заметно меняет decision boundary. Это мотивирует использование ансамблей в следующем семинаре.

## 5. Isolation Forest (обнаружение аномалий)

Isolation Forest - алгоритм обнаружения аномалий, основанный на деревьях. Идея: аномальные точки изолируются (отсекаются от остальных) за меньшее число разбиений, чем нормальные.

In [ ]:
n_samples = 300
n_outliers = 15
X_if, _ = make_blobs(n_samples=n_samples - n_outliers, centers=[[0, 0], [3, 3]], cluster_std=0.5, random_state=42)

rng = np.random.RandomState(42)
X_outliers = rng.uniform(low=-4, high=8, size=(n_outliers, 2))
X_if = np.vstack([X_if, X_outliers])

clf_if = IsolationForest(contamination=n_outliers / n_samples, random_state=42)
y_pred_if = clf_if.fit_predict(X_if)

plt.figure(figsize=(10, 7))
plt.scatter(X_if[:, 0], X_if[:, 1], c=y_pred_if, cmap='coolwarm')
plt.colorbar(label='Выбросы (-1) vs. Нормальные точки (1)')
plt.title('Обнаружение выбросов с помощью Isolation Forest')
plt.grid(True)
plt.show()

### Пример: обнаружение мошеннических транзакций

In [ ]:
n_transactions = 10000
n_frauds = 100

normal_transactions = np.random.randn(n_transactions - n_frauds, 2)
normal_transactions[:, 0] = np.abs(normal_transactions[:, 0]) * 100
normal_transactions[:, 1] = np.abs(normal_transactions[:, 1]) * 10

fraud_transactions = np.random.randn(n_frauds, 2)
fraud_transactions[:, 0] = np.abs(fraud_transactions[:, 0]) * 500 + 200
fraud_transactions[:, 1] = np.abs(fraud_transactions[:, 1]) * 3 + 20

X_fraud = np.vstack([normal_transactions, fraud_transactions])
true_labels = np.zeros(n_transactions)
true_labels[n_transactions - n_frauds:] = 1

scaler = StandardScaler()
X_fraud_scaled = scaler.fit_transform(X_fraud)

clf_fraud = IsolationForest(contamination=n_frauds / n_transactions, random_state=42)
y_pred_fraud = clf_fraud.fit_predict(X_fraud_scaled)
y_pred_fraud = [1 if i == -1 else 0 for i in y_pred_fraud]

print("Матрица ошибок:")
print(confusion_matrix(true_labels, y_pred_fraud))
print("\nОтчет о классификации:")
print(classification_report(true_labels, y_pred_fraud))

plt.figure(figsize=(12, 8))
plt.scatter(X_fraud[:, 0], X_fraud[:, 1], c=y_pred_fraud, cmap='coolwarm', alpha=0.7)
plt.xlabel('Сумма транзакции')
plt.ylabel('Время суток')
plt.title('Обнаружение мошеннических транзакций с помощью Isolation Forest')
plt.colorbar(label='Мошенничество (1) vs. Нормальная транзакция (0)')
plt.grid(True)
plt.show()

## 6. Инкрементальные деревья

Hoeffding Tree - дерево решений для потоковых данных. Обучается инкрементально (по одному примеру), может адаптироваться к concept drift (изменению распределения данных).

In [ ]:
from river import tree as river_tree
from river import metrics as river_metrics

def generate_drift_stream(n_samples=10000, n_features=2, n_classes=2, random_state=42):
    rng = np.random.RandomState(random_state)
    X = np.zeros((n_samples, n_features))
    y = np.zeros(n_samples)
    for i in range(n_samples):
        if i < n_samples / 3:
            center = 0
        elif i < 2 * n_samples / 3:
            center = 2
        else:
            center = 4
        if rng.rand() > 0.5:
            X[i, 0] = rng.normal(center, 1)
            X[i, 1] = rng.normal(0, 1)
            y[i] = 0
        else:
            X[i, 0] = rng.normal(0, 1)
            X[i, 1] = rng.normal(center, 1)
            y[i] = 1
    stream = [({'x0': X[i, 0], 'x1': X[i, 1]}, y[i]) for i in range(n_samples)]
    return stream

stream = generate_drift_stream()

In [ ]:
model = river_tree.HoeffdingTreeClassifier(
    grace_period=100,
    split_criterion='entropy',
    max_depth=5,
)

metric = river_metrics.Accuracy()
results = []

for i, (x, y_val) in enumerate(stream):
    y_pred = model.predict_one(x)
    if i > 0:
        metric.update(y_val, y_pred)
        results.append((i, metric.get()))
    model.learn_one(x, y_val)
    if i % 1000 == 0 and i > 0:
        print(f"Наблюдение {i}, Точность: {metric.get():.4f}")

results_df = pd.DataFrame(results, columns=['Наблюдение', 'Точность'])

In [ ]:
# Сравнение с обычным деревом (переобучение на батчах)
batch_size = 1000
batches = [stream[i:i + batch_size] for i in range(0, len(stream), batch_size)]

batch_results = []
for batch_idx, batch in enumerate(batches[:-1]):
    X_batch_train = pd.DataFrame([x for x, _ in batch])
    y_batch_train = np.array([y_val for _, y_val in batch])
    test_batch = batches[batch_idx + 1]
    X_batch_test = pd.DataFrame([x for x, _ in test_batch])
    y_batch_test = np.array([y_val for _, y_val in test_batch])
    batch_model = DecisionTreeClassifier(max_depth=5, random_state=42)
    batch_model.fit(X_batch_train, y_batch_train)
    accuracy = batch_model.score(X_batch_test, y_batch_test)
    batch_results.append(((batch_idx + 1) * batch_size, accuracy))
    print(f"Батч {batch_idx + 1}, Точность обычного дерева: {accuracy:.4f}")

plt.figure(figsize=(12, 6))
plt.plot(results_df['Наблюдение'], results_df['Точность'], label='Инкрементальное дерево')
plt.plot(*zip(*batch_results), 'o-', label='Обычное дерево (переобучение на батчах)')
plt.xlabel('Количество наблюдений')
plt.ylabel('Точность')
plt.title('Сравнение инкрементального и обычного деревьев решений')
plt.legend()
plt.grid(True)
plt.show()

## 7. Применение: кредитный скоринг

Деревья решений особенно ценны в задачах, где важна интерпретируемость: кредитный скоринг, медицинская диагностика, юридические решения.

In [ ]:
np.random.seed(42)
n_samples = 1000

age = np.random.normal(35, 10, n_samples)
income = np.exp(np.random.normal(10, 0.7, n_samples)) / 1000
credit_history = np.random.choice([0, 0.5, 1], n_samples, p=[0.2, 0.3, 0.5])
employment_years = np.random.gamma(2, 3, n_samples)
debt_to_income = np.random.beta(2, 5, n_samples)

credit_approval = (
    0.2 * age + 0.4 * income + 0.25 * credit_history
    + 0.1 * employment_years - 0.3 * debt_to_income
    + np.random.normal(0, 0.1, n_samples)
)
credit_approval = (credit_approval > np.median(credit_approval)).astype(int)

credit_data = pd.DataFrame({
    'возраст': age,
    'доход': income,
    'кредитная_история': credit_history,
    'стаж_работы': employment_years,
    'соотношение_долга_к_доходу': debt_to_income,
    'одобрен_кредит': credit_approval,
})

X_credit = credit_data.drop('одобрен_кредит', axis=1)
y_credit = credit_data['одобрен_кредит']
X_cr_train, X_cr_test, y_cr_train, y_cr_test = train_test_split(
    X_credit, y_credit, test_size=0.3, random_state=42,
)

In [ ]:
clf_credit = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_credit.fit(X_cr_train, y_cr_train)

y_cr_pred = clf_credit.predict(X_cr_test)

print(f"Точность: {accuracy_score(y_cr_test, y_cr_pred):.4f}")
print("\nОтчет о классификации:")
print(classification_report(y_cr_test, y_cr_pred))
print("\nМатрица ошибок:")
print(confusion_matrix(y_cr_test, y_cr_pred))

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(clf_credit, filled=True, feature_names=X_credit.columns,
          class_names=['Отказ', 'Одобрено'], rounded=True)
plt.title("Дерево решений для кредитного скоринга")
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'Признак': X_credit.columns,
    'Важность': clf_credit.feature_importances_,
}).sort_values('Важность', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Признак'], feature_importance['Важность'])
plt.xlabel('Важность')
plt.title('Важность признаков в модели кредитного скоринга')
plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Примеры правил принятия решений
print("Примеры правил принятия решений:")
tree_ = clf_credit.tree_
node_indicator = clf_credit.decision_path(X_cr_test)
leaf_id = clf_credit.apply(X_cr_test)

for sample_id in range(5):
    print(f"\nПример {sample_id + 1}:")
    node_index = node_indicator.indices[
        node_indicator.indptr[sample_id] : node_indicator.indptr[sample_id + 1]
    ]
    for node_id in node_index:
        if leaf_id[sample_id] == node_id:
            decision = 'Одобрено' if clf_credit.predict([X_cr_test.iloc[sample_id]])[0] == 1 else 'Отказ'
            print(f"  -> Решение: {decision}")
            continue
        if X_cr_test.iloc[sample_id, tree_.feature[node_id]] <= tree_.threshold[node_id]:
            threshold_sign = "<="
        else:
            threshold_sign = ">"
        feature = X_credit.columns[tree_.feature[node_id]]
        threshold = tree_.threshold[node_id]
        print(f"  {feature} {threshold_sign} {threshold:.2f}")